In [1]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

RAW = Path("../data/raw")
SOURCES = {   # file -> readable title (used in citations later)
    "labour_law.pdf":                "UAE Labour Law (Federal Decree-Law No. 33 of 2021)",
    "exec_regs.pdf":                 "Executive Regulation (Cabinet Resolution No. 1 of 2022)",
    "labour_law_amendment_2022.pdf": "Labour Law Amendment (Federal Decree-Law No. 14 of 2022)",
    "pdpl.pdf":                      "UAE PDPL (Federal Decree-Law No. 45 of 2021)",
}

pages_by_doc = {}
for fname in SOURCES:
    pages = PyPDFLoader(str(RAW / fname)).load()     # one Document per PDF page
    pages_by_doc[fname] = pages
    total = sum(len(p.page_content) for p in pages)
    print(f"{fname:30} {len(pages):3} pages  {total:>9,} chars")

/var/folders/0w/lhff5nrj5jgc5m4mkwnkg5g00000gn/T/ipykernel_54236/3139329251.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


labour_law.pdf                  46 pages     87,616 chars
exec_regs.pdf                   31 pages     57,957 chars
labour_law_amendment_2022.pdf    2 pages      1,667 chars
pdpl.pdf                        29 pages     44,032 chars


In [2]:
doc = pages_by_doc["labour_law.pdf"]
for i in [0, 5]:
    print(f"===== PAGE {i} =====")
    print(doc[i].page_content[:1200])

===== PAGE 0 =====
Federal Decree by Law No. (33) of 2021 Concerning Regulating Labor Relations 1 
Federal Decree by Law No. (33) of 2021 
Regulating Labor Relations 
 
We Khalifa bin Zayed Al Nahyan                      President of the United Arab Emirates, 
− Upon reviewing the Constitution; 
− Federal Law No. (1) of 1972 Concerning the Jurisdictions of Ministries and the Powers of 
Ministers, and any amendments thereto; 
− Federal Law No. (6) of 1973 Concerning the Entry and Residence of Foreigners, and any 
amendments thereto; 
− Federal Law No. (8) of 1980 Regulating the Labor Relations, and any amendments 
thereto; 
− Federal Law No. (3) of 1987 Promulgating the Penal Code, and any amendments thereto; 
− Federal Law No. (10) of 1992 Promulgating the Law of Evidence of Civil Transactions Law, 
and any amendments thereto; 
− Federal Law No. (11) of 1992 Promulgating the Civil Procedures Code, and any 
amendments thereto; 
− Federal Law No. (35) of 1992 Promulgating the Criminal Pr

In [3]:
import re
from collections import Counter

def norm(line):
    """Drop a trailing page number so 'Header ... 1' and 'Header ... 2' count as the same line."""
    return re.sub(r"\s*\d+\s*$", "", line.strip())

def find_repeating_lines(pages, min_share=0.5):
    """Lines (ignoring page numbers) that appear on at least half the pages = headers/footers."""
    counts = Counter()
    for p in pages:
        counts.update({norm(l) for l in p.page_content.splitlines() if l.strip()})
    limit = max(3, len(pages) * min_share)
    return {line for line, c in counts.items() if line and c >= limit}

def clean_pages(pages):
    repeating = find_repeating_lines(pages)
    kept = []
    for p in pages:
        for line in p.page_content.splitlines():
            s = line.strip()
            if not s or norm(s) in repeating:                              # header/footer
                continue
            if re.fullmatch(r"(page\s*)?\d+(\s*(of|/)\s*\d+)?", s, re.I):  # bare page numbers
                continue
            kept.append(s)
    text = "\n".join(kept)                          # all pages -> one text
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)    # re-join hyphen-split words
    text = re.sub(r"[ \t]+", " ", text)             # collapse spaces
    return text, repeating

clean = {}
for fname, pages in pages_by_doc.items():
    text, removed = clean_pages(pages)
    clean[fname] = text
    raw_len = sum(len(p.page_content) for p in pages)
    print(f"{fname:30} {raw_len:>7,} -> {len(text):>7,} chars | removed: {sorted(removed)[:2]}")

labour_law.pdf                  87,616 ->  82,351 chars | removed: ['Federal Decree by Law No. (33) of 2021 Concerning Regulating Labor Relations']
exec_regs.pdf                   57,957 ->  53,018 chars | removed: ['Cabinet Resolution of 2022 Concerning the Executive Regulations of Federal Decree-Law of 2021 Regulating Labour', 'Relations']
labour_law_amendment_2022.pdf    1,667 ->   1,605 chars | removed: []
pdpl.pdf                        44,032 ->  41,127 chars | removed: ['Federal Decree by Law Concerning the Protection of Personal Data']


In [4]:
from langchain_core.documents import Document

OUT = Path("../data/clean"); OUT.mkdir(exist_ok=True)
docs = []
for fname, text in clean.items():
    (OUT / fname.replace(".pdf", ".txt")).write_text(text, encoding="utf-8")
    docs.append(Document(page_content=text, metadata={"source": fname, "title": SOURCES[fname]}))
    heads = re.findall(r"(?m)^Article\b.*$", text)          # lines that START with "Article"
    print(f"\n{fname:30} lines starting with 'Article': {len(heads)}")
    print("   first 4:", heads[:4])


labour_law.pdf                 lines starting with 'Article': 76
   first 4: ['Article (1)', 'Article )2 (', 'Article )3 (', 'Article )4(']

exec_regs.pdf                  lines starting with 'Article': 40
   first 4: ['Article (1)', 'Article (2)', 'Article (3)', 'Article (4)']

labour_law_amendment_2022.pdf  lines starting with 'Article': 3
   first 4: ['Article (1)', 'Article (2)', 'Article (3)']

pdpl.pdf                       lines starting with 'Article': 31
   first 4: ['Article (1)', 'Article (2)', 'Article (3)', 'Article (4)']
